# Findings 

- `self.param_groups` is inherited from `optim.Optimizer`


In [2]:
import torch 
import torch.optim as optim
from torch.optim.optimizer import required 

# Layer-wise Adaptive Rate Scaling

In [ ]:


class LARS(optim.Optimizer): 
    def __init__(
        self,
        params,
        lr = required, 
        momentum = 0, 
        dampening = 0, 
        weight_decay = 0, 
        nestrov = False,
        eta = 1e-3,
        eps = 1e-8,
        clip_lr = False,
        exclude_bias_n_norm = False, 
    ):
        if lr is not required and lr < 0.0 : 
            raise ValueError(f'Invalid learning rate: {lr}')
        if momentum < 0.0 : 
            raise ValueError(f"Invalid momentum values: {momentum}")
        if weight_decay < 0.0 : 
            raise ValueError(f"Invalid weight_decay values: {weight_decay}")

        defaults = dict(
            lr = lr, 
            momentum = momentum,
            dampening = dampening,
            weight_decay = weight_decay,
            nestrov = nestrov, 
            eta = eta, 
            eps = eps,
            clip_lr = clip_lr, 
            exclude_bias_n_norm = exclude_bias_n_norm
        )

        if nestrov and (momentum <= 0 or dampening != 0): 
            raise ValueError("Nestrov momentum requires a momentum and zero dampening")

        super().__init__(params,defaults)     

    def __setstate__(self,state):
        super().__setstate__(state)

        for group in self.param_groups:
            group.setdefault("nestrov",False)

    
    @torch.no_grad()
    def step(self,closure = None):
        """Performs a single optimization step. 
        Args : 
            closure (callable, optional): A closure that reevaluates the model 
                and returns the loss 
        """
        loss = None 
        if closure is not None:
            with torch.enable_grad(): 
                loss = closure()

        # exlude scaling for params with 0 weight decay 
        for group in self.param_groups:
            weight_decay = group["weight_decay"]
            momentum = group["momentum"]
            dampening = group["dampening"]
            nestrov = group["nestrov"]

            for p in group["params"]:
                if p.grad is None : 
                    continue 

                d_p = p.grad
                p_norm = torch.norm(p.data)
                g_norm = torch.norm(p.grad.data)

                # lars scaling + weight decay part 
                if p.ndim != 1 or not group["exclude_bias_n_norm"]: # it doesn't make sense to use LARS on ndim == 1, 
                    if p_norm != 0 and g_norm != 0:
                        lars_lr = p_norm / (
                            g_norm + p_norm * weight_decay + group["eps"]
                        )
                        lars_lr *= group["eta"]

                        # clip lr 
                        if group["clip_lr"]: 
                            lars_lr = min(lars_lr / group["lr"],1)
                        
                        d_p = d_p.add(p, alpha = weight_decay)  # d_p​ ← d_p ​ + λ⋅p
                        d_p *= lars_lr

                # sgd part 
                if momentum != 0 : 
                    param_state = self.state[p]
                    if "momentum_buffer" not in param_state:
                        buf = param_state["momentum_buffer"] = torch.clone(d_p).detach()
                    else : 
                        buf = param_state["momentum_buffer"]
                        buf.mul_(momentum).add_(d_p, alpha= 1 - dampening)
                    if nestrov : 
                        d_p = d_p.add(buf,alpha = momentum)
                    else : 
                        d_p = buf 

                p.add_(d_p,alpha=-group["lr"])

        return loss 


In [ ]:
optimizer = LARS(
    [{"params": model.parameters(), "lr": 0.3},
     {"params": linear_probe.parameters(), "lr": 0.1}],
    weight_decay=1e-4, eta=0.02, clip_lr=True, exclude_bias_n_norm=True, momentum=0.9,
)

# WarmupCosineScheduler 

In [1]:
class WarmupCosineScheduler: 
    """Warmup cosine learning rate scheduler"""

    def __init__(
        self,
        optimizer,
        warmup_epochs,
        max_epochs,
        base_lr,
        min_lr = 0.0,
        warmup_start_lr = 3e-5
    ):

        self.optimizer = optimizer 
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.warmup_start_lr = warmup_start_lr 


    def step(self,epoch):
        if epoch < self.warmup_epochs:
            lr = self.warmup_start_lr + epoch * (
                self.base_lr - self.warmup_start_lr
            ) / (self.warmup_epochs - 1)

        else : 
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
                1 
                + torch.cos(
                    torch.tensor(
                        (epoch - self.warmup_epochs)
                        / (self.max_epochs - self.warmup_epochs)
                        * 3.14159
                    )
                )
            )

        for param_group in self.optimizer.param_groups: 
            param_group["lr"] = lr 